# 01c · Multiclasse e sbilanciamento

Nei notebook precedenti abbiamo visto modelli con il 99% di accuratezza che mancano metà degli attacchi più rari, e detector di anomalie che trovano molti attacchi solo accettando molti falsi allarmi. Qui mettiamo ordine:
- perché **l'accuratezza** inganna quando gli attacchi sono pochi;
- quali **metriche** usare al suo posto, e come confrontare modelli con le **curve**;
- come la **soglia** di decisione sposta il compromesso tra attacchi trovati e falsi allarmi;
- cosa cambia se chiediamo al modello di riconoscere **il tipo** di attacco, e come aiutarlo con le **classi rare**.

**Come usare il notebook.** Esegui le celle in ordine con `Shift + Invio`. I riquadri **Prova tu** sono piccole modifiche da fare in aula. Le **Sfide** in fondo sono da fare a casa, e l'ultima è una gara.

## 0. Setup e dati

In [ ]:
# Cella di setup: eseguila per prima, ogni volta che apri il notebook.
import subprocess, sys
from pathlib import Path

REPO = "salvatore-drago/nids-ml-lab"

if "google.colab" in sys.modules:
    root = Path("/content/nids-ml-lab")
    if not root.exists():
        url = f"https://github.com/{REPO}.git"
        try:  # repository privato: token nei Secrets di Colab, con nome GITHUB_TOKEN
            from google.colab import userdata
            url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/{REPO}.git"
        except Exception:
            pass
        if subprocess.run(["git", "clone", "-q", "--depth", "1", url, str(root)]).returncode:
            raise RuntimeError("Clone del repository non riuscito: controlla la connessione o l'accesso.")
    else:
        subprocess.run(["git", "-C", str(root), "pull", "-q"])
else:
    root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "nidslab").is_dir()), None)
    if root is None:
        raise RuntimeError("Apri il notebook dalla cartella notebooks/ del repository nids-ml-lab.")

if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
import nidslab as nl

nl.ensure_packages()
nl.set_style()
print("Repository:", root)

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, IsolationForest, RandomForestClassifier
from sklearn.metrics import (accuracy_score, average_precision_score, classification_report,
                             f1_score, precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

SEED = 42

Stessi dati e stessa divisione del notebook `01b`: 20% addestramento, 10% validazione, 70% test. In questo notebook **guardiamo sempre la validazione**; il test lo userà solo la gara, alla fine.

In [ ]:
data, features = nl.prepare_static()
X, y = data[features], data["label_binary"]
lab = data["label"].astype(str)

X_train, X_rest, y_train, y_rest, lab_train, lab_rest = train_test_split(
    X, y, lab, train_size=0.20, stratify=lab, random_state=SEED)
X_val, X_test, y_val, y_test, lab_val, lab_test = train_test_split(
    X_rest, y_rest, lab_rest, train_size=0.125, stratify=lab_rest, random_state=SEED)
print(f"Addestramento: {len(X_train):,}   Validazione: {len(X_val):,}   Test: {len(X_test):,}")

## 1. Il paradosso dell'accuratezza

### 1.1 Il modello pigro

Un "modello" che risponde sempre **benigno**, qualunque cosa veda. Non ha imparato niente.

In [ ]:
lazy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
p = lazy.predict(X_val)
print(f"Accuratezza:           {accuracy_score(y_val, p):.2%}")
print(f"Attacchi riconosciuti: {p[y_val == 1].mean():.2%}")

> **Da notare.** Un modello che non ha imparato niente ottiene quasi il **90% di accuratezza** e non trova nemmeno un attacco. La sua accuratezza è sempre uguale alla quota della classe più frequente: qui i benigni sono il 90% dei flussi.

### 1.2 Quanti allarmi sono veri?

Prendiamo ora un modello buono: il gradient boosting, il migliore di `01b`. Riconosce quasi tutti gli attacchi e fa pochissimi falsi allarmi. Ma quando suona un allarme, **qual è la probabilità che sia un attacco vero?** Questa quota si chiama **precisione** (in inglese *precision*: attenzione a non confonderla con l'accuratezza), e non dipende solo dal modello: dipende anche da **quanti attacchi ci sono nel traffico** (il *tasso di base*).

Se $r$ è la quota di attacchi riconosciuti, $f$ la quota di falsi allarmi sui benigni e $b$ il tasso di base:

$$\text{precisione} = \frac{r \cdot b}{r \cdot b + f \cdot (1 - b)}$$

Nel nostro dataset gli attacchi sono il 10% dei flussi. In una rete reale sono molti meno.

In [ ]:
gb = HistGradientBoostingClassifier(random_state=SEED).fit(X_train, y_train)
p_gb = gb.predict_proba(X_val)[:, 1]
pred_gb = (p_gb >= 0.5).astype(int)
r = pred_gb[y_val == 1].mean()
f = pred_gb[y_val == 0].mean()
print(f"Gradient boosting: attacchi riconosciuti {r:.2%}, falsi allarmi {f:.3%}")

> **Da notare.** È lo stesso gradient boosting di `01b`: riconosce il 98,6% degli attacchi con lo 0,085% di falsi allarmi.

> **Prova tu.** Cambia `BASE_RATE` (per esempio 0.5, 0.001, 0.00001) e riesegui. Con quanti attacchi nel traffico la precisione scende sotto il 50%?

In [ ]:
BASE_RATE = 0.0001
FLOWS_PER_DAY = 1_000_000

rows = []
for b in sorted({0.10, 0.01, 0.001, 0.0001, BASE_RATE}, reverse=True):
    precision = r * b / (r * b + f * (1 - b))
    rows.append({
        "tasso di base": f"{b:.4%}",
        "attacchi su 1 milione di flussi": round(b * FLOWS_PER_DAY),
        "allarmi veri": round(r * b * FLOWS_PER_DAY),
        "falsi allarmi": round(f * (1 - b) * FLOWS_PER_DAY),
        "precisione": f"{precision:.1%}",
    })
pd.DataFrame(rows)

> **Da notare.**
> - Con il 10% di attacchi, come nel nostro dataset, il 99,2% degli allarmi è vero: il modello sembra perfetto.
> - Con un attacco ogni mille flussi, gli allarmi veri e quelli falsi sono quasi alla pari (986 contro 848).
> - Con un attacco ogni diecimila flussi, **nove allarmi su dieci sono falsi**: 99 veri contro 848 falsi.
> - Il modello non è cambiato: attacchi riconosciuti e tasso di falsi allarmi sono gli stessi in tutte le righe. È cambiato solo il traffico.

È la **base-rate fallacy** applicata al rilevamento delle intrusioni (Axelsson, 2000): quando gli attacchi sono rari, anche un tasso di falsi allarmi piccolissimo produce più allarmi falsi che veri. Gli operatori, sommersi di allarmi inutili, smettono di fidarsi del sistema, e un allarme vero rischia di essere ignorato insieme agli altri.

## 2. Le metriche giuste

### 2.1 Dalla matrice di confusione

Tutto parte dai quattro numeri della matrice di confusione:
- **veri positivi** (TP): attacchi segnalati;
- **falsi negativi** (FN): attacchi mancati;
- **falsi positivi** (FP): benigni segnalati, cioè falsi allarmi;
- **veri negativi** (TN): benigni lasciati passare.

| Metrica | Nome inglese | Formula | Domanda a cui risponde |
|---|---|---|---|
| Quota di attacchi riconosciuti | *recall*, *detection rate*, *true positive rate* | TP / (TP + FN) | Quanti attacchi trovo? |
| Precisione | *precision* | TP / (TP + FP) | Quanti allarmi sono veri? |
| Tasso di falsi allarmi | *false positive rate* (FPR) | FP / (FP + TN) | Quanti benigni disturbo? |
| F1 | *F1 score* | media armonica di precisione e recall | Un compromesso tra le due |
| Accuratezza | *accuracy* | (TP + TN) / tutti | Quante risposte giuste in totale? |

In [ ]:
tp = int(((pred_gb == 1) & (y_val == 1)).sum())
fn = int(((pred_gb == 0) & (y_val == 1)).sum())
fp = int(((pred_gb == 1) & (y_val == 0)).sum())
tn = int(((pred_gb == 0) & (y_val == 0)).sum())
print(f"TP {tp:,}   FN {fn:,}   FP {fp:,}   TN {tn:,}\n")
print(f"Recall:      {tp / (tp + fn):.4f}")
print(f"Precisione:  {tp / (tp + fp):.4f}")
print(f"FPR:         {fp / (fp + tn):.4%}")
print(f"F1:          {f1_score(y_val, pred_gb):.4f}")
print(f"Accuratezza: {accuracy_score(y_val, pred_gb):.4f}")

> **Da notare.** Sulla validazione il gradient boosting sbaglia solo **18 flussi benigni e 35 attacchi** su oltre 23 mila. La precisione misurata (0,9925) coincide con la riga del 10% della tabella della sezione 1.2: formula e misura raccontano la stessa cosa.

### 2.2 Le curve: tutte le soglie insieme

Un modello che dà un **punteggio** (una probabilità, o un grado di anomalia) non ha una sola matrice di confusione: ne ha una per ogni soglia. Le curve le mostrano tutte insieme:
- la **curva ROC** mette sull'asse orizzontale il tasso di falsi allarmi e su quello verticale la recall;
- la **curva precision-recall** mette la recall in orizzontale e la precisione in verticale.

Confrontiamo il gradient boosting con i due detector di anomalie di `01b`, addestrati come lì solo sul traffico benigno. Per loro, il punteggio è il grado di anomalia.

In [ ]:
def log1p_positive(X):
    return np.log1p(np.clip(X, 0, None))


benign_train = X_train[y_train == 0]
t0 = time.perf_counter()
iso = IsolationForest(n_estimators=200, n_jobs=-1, random_state=SEED).fit(benign_train)
lof = make_pipeline(FunctionTransformer(log1p_positive), StandardScaler(),
                    LocalOutlierFactor(n_neighbors=20, novelty=True, n_jobs=-1)).fit(benign_train)

# Per i detector, un punteggio più alto deve voler dire "più anomalo": invertiamo il segno.
scores = {
    "Gradient boosting": p_gb,
    "Isolation Forest": -iso.decision_function(X_val),
    "LOF": -lof.decision_function(X_val),
}
print(f"Detector addestrati e valutati in {time.perf_counter() - t0:.0f} s")

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(12, 4.8))
for name, s in scores.items():
    fpr_c, tpr_c, _ = roc_curve(y_val, s)
    prec_c, rec_c, _ = precision_recall_curve(y_val, s)
    ax_roc.plot(fpr_c, tpr_c, label=f"{name} (AUC {roc_auc_score(y_val, s):.3f})")
    ax_pr.plot(rec_c, prec_c, label=f"{name} (AP {average_precision_score(y_val, s):.3f})")

ax_roc.plot([0, 1], [0, 1], color=nl.INK_SECONDARY, lw=1, ls="--", label="Caso")
ax_roc.set_xlabel("Tasso di falsi allarmi")
ax_roc.set_ylabel("Attacchi riconosciuti (recall)")
ax_roc.set_title("Curva ROC", loc="left")
ax_roc.legend(loc="best")

ax_pr.axhline(y_val.mean(), color=nl.INK_SECONDARY, lw=1, ls="--", label="Caso")
ax_pr.set_xlabel("Attacchi riconosciuti (recall)")
ax_pr.set_ylabel("Precisione")
ax_pr.set_title("Curva precision-recall", loc="left")
ax_pr.legend(loc="best")
plt.show()

L'area sotto la curva ROC (AUC) e la *average precision* (AP, che riassume la curva precision-recall) valgono 1 per un modello perfetto. La linea tratteggiata è un modello che tira a caso: per la ROC è la diagonale, per la precision-recall è la quota di attacchi nei dati.

La ROC guarda i falsi allarmi **in proporzione ai benigni**, che sono tantissimi: un tasso di falsi allarmi dell'1% sembra minuscolo sull'asse, ma sono centinaia di flussi. La precision-recall guarda i falsi allarmi **in proporzione agli allarmi**, ed è molto più severa quando gli attacchi sono pochi.

> **Da notare.**
> - Nella ROC i due detector di anomalie sembrano buoni: AUC 0,91. Nella precision-recall la loro AP è 0,47, e nel punto migliore (circa il 70-75% di attacchi riconosciuti) la precisione non supera 0,6-0,67: **almeno un allarme su tre è falso**, e questo con il 10% di attacchi nel traffico.
> - Tutta la zona che interessa a un IDS, sotto l'1% di falsi allarmi, nella ROC è schiacciata in una striscia sottilissima contro l'asse verticale.
> - Il gradient boosting è quasi perfetto in entrambe (AUC 1,000, AP 0,999). Un risultato così dovrebbe **insospettire**: è la domanda 1 del notebook `01a`, e nella seconda esercitazione vedremo cosa succede quando gli attacchi da riconoscere non sono quelli visti in addestramento.

## 3. Spostare la soglia

Il gradient boosting restituisce una probabilità di attacco; finora l'abbiamo trasformata in una decisione con la soglia 0,5. Ma la soglia la scegliamo noi, ed è la manopola della sensibilità: più bassa trova più attacchi e disturba di più, più alta disturba meno e lascia passare di più.

> **Prova tu.** Muovi lo slider. Dove metteresti la soglia per un sistema che un operatore deve controllare a mano ogni giorno? Se lo slider non compare, cambia `THRESHOLD` ed esegui di nuovo la cella.

In [ ]:
THRESHOLD = 0.5


def show_threshold(threshold):
    pred = (p_gb >= threshold).astype(int)
    rec = pred[y_val == 1].mean()
    fpr = pred[y_val == 0].mean()
    missed = int(((pred == 0) & (y_val == 1)).sum())
    alarms_false = int(((pred == 1) & (y_val == 0)).sum())
    print(f"Soglia {threshold:.2f}")
    print(f"  attacchi riconosciuti: {rec:.2%}   ({missed} attacchi mancati su {int((y_val == 1).sum())})")
    print(f"  falsi allarmi:         {fpr:.3%}   ({alarms_false} su {int((y_val == 0).sum())} flussi benigni)")
    print(f"  su 1 milione di flussi benigni al giorno: circa {fpr * 1_000_000:,.0f} falsi allarmi")


try:
    import ipywidgets as widgets
    widgets.interact(show_threshold, threshold=widgets.FloatSlider(
        value=THRESHOLD, min=0.01, max=0.99, step=0.01, continuous_update=False, description="Soglia"))
except Exception:
    show_threshold(THRESHOLD)

Invece di cercare la soglia a mano, si può partire dal vincolo: "non posso permettermi più dello 0,05% di falsi allarmi". La soglia giusta è quella che lascia sopra di sé solo lo 0,05% dei flussi benigni di **validazione**.

In [ ]:
TARGET_FPR = 0.0005

# Ordiniamo i punteggi dei benigni dal più alto al più basso: possiamo permetterci
# al massimo k falsi allarmi, quindi la soglia va appena sopra il (k+1)-esimo punteggio.
benign_scores = np.sort(p_gb[y_val == 0])[::-1]
k = int(TARGET_FPR * len(benign_scores))
threshold = float(np.nextafter(benign_scores[k], 1))
print(f"Al massimo {TARGET_FPR:.2%} di falsi allarmi, cioè {k} flussi benigni su {len(benign_scores):,}")
print(f"Soglia: {threshold:.3f}\n")
show_threshold(threshold)

> **Da notare.**
>
> | Soglia | Attacchi mancati | Falsi allarmi | Su 1 milione di flussi benigni |
> |---|---|---|---|
> | 0,50 | 35 su 2.415 | 18 (0,085%) | circa 850 |
> | scelta con `TARGET_FPR = 0.0005` | circa 40 | al massimo 10 (0,047%) | meno di 500 |
>
> Alzando di poco la soglia, qualche attacco mancato in più compra quasi metà dei falsi allarmi. Se il compromesso convenga lo decide chi deve gestire gli allarmi, non il modello.

## 4. Multiclasse: quale attacco?

Un operatore non vuole sapere solo che c'è un attacco, ma **quale**: un DoS si gestisce in modo diverso da un furto di credenziali. Addestriamo una random forest a riconoscere le 14 classi (Benign più 13 attacchi).

Con più classi servono metriche che le riassumano:
- **accuratezza**: quota di flussi classificati nella classe giusta;
- **F1 macro**: la media degli F1 delle classi, dove ogni classe pesa uguale. Una classe rara sbagliata pesa quanto il traffico benigno;
- **F1 pesato**: la media degli F1 pesata per il numero di flussi di ogni classe. Somiglia all'accuratezza.

In [ ]:
ORDER = ["Benign", "DoS_Hulk", "DoS_GoldenEye", "DoS_Slowhttptest", "DoS_Slowloris", "Heartbleed",
         "DDoS_LOIT", "Port_Scan", "FTP-Patator", "SSH-Patator", "Botnet_ARES",
         "Web_Brute_Force", "Web_XSS", "Web_SQL_Injection"]

t0 = time.perf_counter()
rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED).fit(X_train, lab_train)
pred_rf = rf.predict(X_val)
print(f"Addestramento e predizione in {time.perf_counter() - t0:.0f} s\n")
print(f"Accuratezza: {accuracy_score(lab_val, pred_rf):.4f}")
print(f"F1 pesato:   {f1_score(lab_val, pred_rf, average='weighted', zero_division=0):.4f}")
print(f"F1 macro:    {f1_score(lab_val, pred_rf, average='macro', zero_division=0):.4f}\n")
print(classification_report(lab_val, pred_rf, labels=ORDER, digits=3, zero_division=0))

> **Da notare.**
> - Accuratezza e F1 pesato sono al 99,6%; l'**F1 macro è 0,74**. Basta dare lo stesso peso a ogni classe per far emergere quello che l'accuratezza nasconde.
> - Le classi grandi vanno benissimo (DDoS 1,000, DoS Hulk 0,991, Port Scan 0,982); i web attack crollano (F1 tra 0,29 e 0,40).
> - Heartbleed ha F1 zero, ma nella validazione c'è **un solo flusso**: quel singolo zero pesa da solo un quattordicesimo dell'F1 macro. Per le classi con pochissimi esempi, la metrica è più rumore che segnale.

In [ ]:
nl.plot_confusion_matrix(lab_val, pred_rf, labels=ORDER, title="Random forest, 14 classi (validazione)")
plt.show()

> **Da notare.**
> - Le confusioni **tra attacchi** sono poche: la più visibile è tra web attack (il 43% degli XSS viene scambiato per Web Brute Force). Ordinando le classi per famiglia, finiscono vicino alla diagonale.
> - L'errore principale è la **prima colonna**: attacchi classificati come benigni. Due terzi dei Web Brute Force, il 29% degli Slowhttptest, tra il 12% e il 15% di FTP-Patator, SSH-Patator e Botnet.
> - Per un IDS è la distinzione che conta: scambiare un XSS per un Brute Force è un errore lieve, perché l'operatore guarda comunque il server web; scambiare un attacco per traffico normale vuol dire non guardarlo affatto.

### 4.1 Gli attacchi mancati sono frammenti?

Nel notebook `01a` abbiamo visto che molti flussi d'attacco sono frammenti di un solo pacchetto, identici al traffico benigno. Quanti degli attacchi scambiati per benigni sono frammenti?

In [ ]:
pc = data.loc[X_val.index, "packets_count"].to_numpy()
true = lab_val.to_numpy()
attack = true != "Benign"
missed = attack & (pred_rf == "Benign")
found = attack & (pred_rf != "Benign")

print(f"Attacchi scambiati per benigni:  {missed.sum():>5}, di cui da 1 pacchetto: {(pc[missed] == 1).mean():.1%}")
print(f"Attacchi riconosciuti come tali: {found.sum():>5}, di cui da 1 pacchetto: {(pc[found] == 1).mean():.1%}\n")

t = pd.DataFrame({"classe": true, "mancato": missed, "un_pacchetto": pc == 1})[attack]
t["mancato_e_un_pacchetto"] = t["mancato"] & t["un_pacchetto"]
t.groupby("classe")[["mancato", "mancato_e_un_pacchetto"]].sum().astype(int).query("mancato > 0")

> **Da notare.**
> - Tra gli attacchi riconosciuti, i frammenti di un pacchetto sono quasi assenti (0,3%); tra quelli mancati sono più di un terzo (22 su 59). Quando un frammento d'attacco arriva al modello, viene quasi sempre scambiato per traffico benigno.
> - Per gli attacchi di **forza bruta e il botnet**, quasi tutti gli errori sono frammenti: per FTP-Patator lo sono **tutti e 7**. Il modello riconosce i tentativi di login veri e propri, e sbaglia solo sulle code delle connessioni, che dell'attacco non contengono niente.
> - Per i **DoS**, invece, gli errori sono flussi completi: 10 su 11 per Slowhttptest, 11 su 13 per DoS Hulk, tutti per Slowloris. I DoS lenti somigliano a connessioni benigne rimaste aperte e poco attive.
> - Sono due problemi diversi: il primo nasce da come sono stati costruiti i flussi, il secondo è la vera difficoltà del rilevamento.

## 5. Rimedi allo sbilanciamento

Le classi rare hanno pochissimi esempi da cui imparare. Due rimedi classici:
- **pesare le classi** (`class_weight="balanced"`): un errore su una classe rara costa al modello quanto molti errori su una classe frequente;
- **sovracampionare** le classi rare: le si porta ad almeno 500 esempi duplicando quelli che ci sono. Non a parità con il benigno, che vorrebbe dire centinaia di migliaia di righe in più.

**Regola d'oro: il ricampionamento si fa solo sui dati di addestramento, mai su validazione e test.** Validazione e test devono assomigliare al traffico vero, che è sbilanciato.

In [ ]:
MIN_EXAMPLES = 500

counts = lab_train.value_counts()
rare = {c: MIN_EXAMPLES for c, n in counts.items() if n < MIN_EXAMPLES}
X_over, lab_over = RandomOverSampler(sampling_strategy=rare, random_state=SEED).fit_resample(X_train, lab_train)
print(f"Addestramento: {len(X_train):,} righe -> {len(X_over):,} dopo il sovracampionamento")
print(f"Classi portate a {MIN_EXAMPLES} esempi: {sorted(rare)}\n")

variants = {
    "Nessun rimedio": (RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED), X_train, lab_train),
    "Pesi bilanciati": (RandomForestClassifier(n_estimators=100, class_weight="balanced", n_jobs=-1,
                                               random_state=SEED), X_train, lab_train),
    "Sovracampionamento": (RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED), X_over, lab_over),
}

summary, recall_by_class = [], {}
for name, (model, Xt, lt) in variants.items():
    pred = model.fit(Xt, lt).predict(X_val)
    benign = lab_val.to_numpy() == "Benign"
    summary.append({
        "variante": name,
        "F1 macro": f1_score(lab_val, pred, average="macro", zero_division=0),
        "F1 pesato": f1_score(lab_val, pred, average="weighted", zero_division=0),
        "falsi allarmi": (pred[benign] != "Benign").mean(),
    })
    per_class = pd.Series(pred == lab_val.to_numpy()).groupby(lab_val.to_numpy()).mean()
    recall_by_class[name] = per_class

pd.DataFrame(summary).set_index("variante").round(4)

In [ ]:
table = pd.DataFrame(recall_by_class).loc[ORDER]
table.insert(0, "flussi in validazione", lab_val.value_counts().reindex(ORDER))
table.round(3)

Ricorda quanti esempi ha ogni classe rara nella validazione: con 1 o 2 flussi, una riga della tabella cambia da 0 a 1 per un solo flusso.

> **Da notare.**
> - I **pesi bilanciati** fanno quello che promettono: la quota riconosciuta delle classi rare sale molto (Slowhttptest dal 71% al 92%, Botnet dall'86% al 97%, SSH-Patator dall'88% al 96%, Web Brute Force dal 25% al 50%).
> - Eppure l'F1 macro **scende** (da 0,737 a 0,705) e i falsi allarmi **quadruplicano** (dallo 0,16% allo 0,64%, oltre il limite della gara). Spingendo il modello verso le classi rare, circa 130 flussi benigni finiscono classificati come attacchi rari, e per una classe con trenta flussi veri anche venti falsi allarmi dimezzano la precisione. È la base-rate fallacy della sezione 1.2, dentro il multiclasse.
> - Il **sovracampionamento** aiuta poco: duplica gli stessi pochi esempi senza aggiungere informazione. Heartbleed passa da 2 a 500 esempi nell'addestramento, ma sono sempre gli stessi 2.
> - Le differenze di F1 macro sono di circa 0,03, e con le classi più rare un solo flusso sposta l'F1 macro di quasi 0,07: una parte della differenza può essere rumore (sezione 5 di `01b`).
> - I rimedi allo sbilanciamento non risolvono un problema che non è di quantità: se gli esempi rari sono ambigui per come sono costruiti, pesarli di più o duplicarli non li rende meno ambigui.

---

## Sfide (a casa)

Ogni sfida ha una cella con dei `TODO` da completare e un controllo (`assert`) che ti dice se hai impostato le cose nel modo giusto.

### Sfida 1: SMOTE

Invece di duplicare gli esempi rari, **SMOTE** ne crea di nuovi interpolando tra un esempio e i suoi vicini della stessa classe. Usalo al posto di `RandomOverSampler` nella sezione 5 (stesso obiettivo: almeno 500 esempi per classe) e confronta i risultati.

Attenzione: SMOTE cerca per ogni esempio i suoi `k_neighbors` vicini della stessa classe, quindi una classe deve avere più di `k_neighbors` esempi. Quanti esempi di Heartbleed ci sono nell'addestramento? Cosa vuol dire creare esempi "sintetici" partendo da così pochi punti?

In [ ]:
from imblearn.over_sampling import SMOTE

# TODO: sovracampiona X_train, lab_train con SMOTE fino ad almeno MIN_EXAMPLES esempi per le classi rare
X_smote, lab_smote = ..., ...

assert len(X_smote) == len(lab_smote), "Righe e etichette devono avere la stessa lunghezza"
assert pd.Series(lab_smote).value_counts().min() >= MIN_EXAMPLES, "Ogni classe deve avere almeno MIN_EXAMPLES esempi"

# TODO: addestra la random forest su X_smote, lab_smote e confronta con la tabella della sezione 5

### Sfida 2: dalle classi alle famiglie

Le 14 classi si raggruppano in 8 famiglie (colonna `family`: DoS, DDoS, PortScan, BruteForce, WebAttack, Botnet, Heartbleed, Benign). Confronta due strade:
1. prendere le predizioni della random forest a 14 classi della sezione 4 e **raggrupparle** per famiglia;
2. addestrare una random forest **direttamente sulle famiglie**.

Quale ha l'F1 macro migliore sulle famiglie? Le confusioni della matrice 14×14 della sezione 4 spariscono raggruppando?

In [ ]:
family_of = data.drop_duplicates("label").set_index("label")["family"].astype(str).to_dict()
fam_train = lab_train.map(family_of)
fam_val = lab_val.map(family_of)

# TODO: strada 1, le predizioni della sezione 4 (pred_rf) raggruppate per famiglia
fam_pred_grouped = ...

# TODO: strada 2, una random forest addestrata su fam_train, con le predizioni sulla validazione
fam_pred_direct = ...

assert len(fam_pred_grouped) == len(fam_val) and len(fam_pred_direct) == len(fam_val)
print(f"F1 macro, raggruppando:  {f1_score(fam_val, fam_pred_grouped, average='macro'):.4f}")
print(f"F1 macro, diretto:       {f1_score(fam_val, fam_pred_direct, average='macro'):.4f}")

### Sfida 3: la gara

**Obiettivo:** il miglior **F1 macro sulle 14 classi** sul test, con il vincolo di **falsi allarmi non oltre lo 0,5%** dei flussi benigni. Il punteggio lo calcola `nl.challenge_score()`, uguale per tutti.

**Regole:**
- puoi usare qualunque modello, feature e rimedio visti finora, o altri;
- **ogni scelta si fa sulla validazione** (o con la cross-validation sull'addestramento);
- il test si usa **una volta sola**, alla fine, per il punteggio da consegnare;
- consegna il notebook con il punteggio stampato e due righe che spieghino le scelte.

Come riferimento, ecco il punteggio della random forest della sezione 4 sulla **validazione**:

In [ ]:
nl.challenge_score(lab_val, pred_rf);

In [ ]:
# TODO: il tuo modello, scelto sulla validazione
my_model = ...

# TODO: le predizioni sul test, calcolate una volta sola alla fine
pred_test = ...

assert len(pred_test) == len(lab_test), "Serve una predizione per ogni flusso del test"
result = nl.challenge_score(lab_test, pred_test)